In [3]:
import pandas as pd

# Masukan data
data = pd.read_csv('Skyserver_SQL2_27_2018 6_51_39 PM.csv')

# Menampilkan data
print(data.head())

          objid          ra       dec         u         g         r         i  \
0  1.237650e+18  183.531326  0.089693  19.47406  17.04240  15.94699  15.50342   
1  1.237650e+18  183.598370  0.135285  18.66280  17.21449  16.67637  16.48922   
2  1.237650e+18  183.680207  0.126185  19.38298  18.19169  17.47428  17.08732   
3  1.237650e+18  183.870529  0.049911  17.76536  16.60272  16.16116  15.98233   
4  1.237650e+18  183.883288  0.102557  17.55025  16.26342  16.43869  16.55492   

          z  run  rerun  camcol  field     specobjid   class  redshift  plate  \
0  15.22531  752    301       4    267  3.722360e+18    STAR -0.000009   3306   
1  16.39150  752    301       4    267  3.638140e+17    STAR -0.000055    323   
2  16.80125  752    301       4    268  3.232740e+17  GALAXY  0.123111    287   
3  15.90438  752    301       4    269  3.722370e+18    STAR -0.000111   3306   
4  16.61326  752    301       4    269  3.722370e+18    STAR  0.000590   3306   

     mjd  fiberid  
0  549

In [4]:
# Memeriksa data yang kosong
data.isnull().any()

,0
objid,False
ra,False
dec,False
u,False
g,False
r,False
i,False
z,False
run,False
rerun,False


In [5]:
# Membersihkan data
sky = data.drop(['objid','run','rerun','camcol','field','specobjid'],axis = 1)
print(sky.head())

           ra       dec         u         g         r         i         z  \
0  183.531326  0.089693  19.47406  17.04240  15.94699  15.50342  15.22531   
1  183.598370  0.135285  18.66280  17.21449  16.67637  16.48922  16.39150   
2  183.680207  0.126185  19.38298  18.19169  17.47428  17.08732  16.80125   
3  183.870529  0.049911  17.76536  16.60272  16.16116  15.98233  15.90438   
4  183.883288  0.102557  17.55025  16.26342  16.43869  16.55492  16.61326   

    class  redshift  plate    mjd  fiberid  
0    STAR -0.000009   3306  54922      491  
1    STAR -0.000055    323  51615      541  
2  GALAXY  0.123111    287  52023      513  
3    STAR -0.000111   3306  54922      510  
4    STAR  0.000590   3306  54922      512  


In [6]:
# Memisahkan fitur dan target

X = sky.drop('class', axis=1)
y = sky['class']

In [7]:
from sklearn.model_selection import train_test_split

# Pembagian data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
from sklearn.neighbors import KNeighborsClassifier

# Membuat model KNeighborsClassifier
knn_model = KNeighborsClassifier(n_neighbors=3)

# Melatih model NeighborsClassifier
knn_model.fit(X_train, y_train)

KNeighborsClassifier(n_neighbors=3)

In [9]:
from sklearn.tree import DecisionTreeClassifier

# Membuat model Decision Tree
dt_model = DecisionTreeClassifier(random_state=42)

# Melatih model Decision Tree
dt_model.fit(X_train, y_train)

DecisionTreeClassifier(random_state=42)

In [10]:
from sklearn.metrics import accuracy_score, precision_score

# Evaluasi model KNN
y_pred_knn = knn_model.predict(X_test)
print("Akurasi KNN:", accuracy_score(y_test, y_pred_knn))
print("Presisi KNN:",precision_score(y_test, y_pred_knn, average='weighted'))


# Evaluasi model Decision Tree
y_pred_dt = dt_model.predict(X_test)
print("Akurasi Decision Tree:", accuracy_score(y_test, y_pred_dt))
print("Presisi Decision Tree:",precision_score(y_test, y_pred_dt, average='weighted'))

Akurasi KNN: 0.769
Presisi KNN: 0.7448473584774051
Akurasi Decision Tree: 0.986
Presisi Decision Tree: 0.98599640439982


In [11]:
from sklearn.model_selection import GridSearchCV

# Definisikan parameter grid untuk KNN
param_grid_knn = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance'],
    'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute']
}

# Definisikan parameter grid untuk Decision Tree classifier
param_grid_dt = {
    'criterion': ['gini', 'entropy'],
    'splitter': ['best', 'random'],
    'max_depth': [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 5, 10]
}

In [12]:
# Grid Search untuk KNN
grid_search_knn = GridSearchCV(knn_model, param_grid_knn, cv=5, scoring='accuracy')
grid_search_knn.fit(X_train, y_train)

print("Parameter terbaik KNN :", grid_search_knn.best_params_)
print("Skor terbaik KNN :", grid_search_knn.best_score_)

# Grid Search untuk Decision Tree classifier
grid_search_dt = GridSearchCV(dt_model, param_grid_dt, cv=5, scoring='accuracy')
grid_search_dt.fit(X_train, y_train)

print("Parameter terbaik Decision Tree :", grid_search_dt.best_params_)
print("Skor terbaik Decision Tree :", grid_search_dt.best_score_)

Parameter terbaik KNN : {'algorithm': 'auto', 'n_neighbors': 11, 'weights': 'uniform'}
Skor terbaik KNN : 0.8002500000000001
Parameter terbaik Decision Tree : {'criterion': 'gini', 'max_depth': 10, 'min_samples_leaf': 5, 'min_samples_split': 2, 'splitter': 'best'}
Skor terbaik Decision Tree : 0.9887499999999999


In [13]:
from sklearn.model_selection import cross_val_score

# k-fold cross-validation untuk KNN, k = 5
scores_knn = cross_val_score(knn_model, X, y, cv=5)

print("Skor KNN :", scores_knn)
print("Mean KNN :", scores_knn.mean())

# k-fold cross-validation untuk Decision Tree classifier, k =5
scores_dt = cross_val_score(dt_model, X, y, cv=5)

print("Skor Decision Tree :", scores_dt)
print("Mean Decision Tree :", scores_dt.mean())

Skor KNN : [0.78   0.7535 0.711  0.7435 0.761 ]
Mean KNN : 0.7498
Skor Decision Tree : [0.986  0.982  0.9865 0.9795 0.987 ]
Mean Decision Tree : 0.9842000000000001


In [14]:
# Membuat data baru
new_data = [
    [153.823268, 0.104517, 17.25029, 14.16512, 16.63889, 12.52592, 16.39326, 0.002260, 5316, 52962, 482],
]

# Melakukan prediksi menggunakan model yang telah dilatih
prediksi_dt = dt_model.predict(new_data)


# Menginterpretasikan output dari model
print("Model memprediksi kelas:", prediksi_dt)

Model memprediksi kelas: ['STAR']


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
